# Swiss Legal Retrieval — MILCO Submission Notebook

Offline Kaggle notebook. No internet at runtime.

**Pre-requisites (upload as Kaggle datasets):**
- MILCO model weights → dataset `milco-650m`
- Pre-built sparse index → dataset `swiss-legal-sparse-index`
  - `corpus_sparse.npz` + `corpus_citations.npy`
  - Build locally with `python src/indexer.py`, then upload

In [ ]:
!pip install transformers scipy -q

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from pathlib import Path
from transformers import AutoModel

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/llm-agentic-legal-information-retrieval')
MODEL_DIR  = Path('/kaggle/input/milco-650m')
INDEX_DIR  = Path('/kaggle/input/swiss-legal-sparse-index')
OUTPUT_DIR = Path('/kaggle/working')

# Config
RETRIEVAL_TOP_K    = 100
ADAPTIVE_GAP_FRAC  = 0.15   # tune this on val.csv

In [ ]:
# ── Load test queries ─────────────────────────────────────────────────────
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Test queries: {len(test_df)}')

In [ ]:
# ── Load MILCO model ──────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading MILCO from {MODEL_DIR} on {device} ...')
model = AutoModel.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
model = model.to(device)
model.eval()
print('Model ready')

In [ ]:
# ── Load pre-built sparse index ───────────────────────────────────────────
print('Loading sparse index ...')
index = sp.load_npz(str(INDEX_DIR / 'corpus_sparse.npz'))
citations = np.load(str(INDEX_DIR / 'corpus_citations.npy'), allow_pickle=True).tolist()
print(f'Index: {index.shape[0]:,} docs, vocab={index.shape[1]:,}, nnz={index.nnz:,}')

In [ ]:
# ── Encode queries ────────────────────────────────────────────────────────
queries = test_df['query'].tolist()
print(f'Encoding {len(queries)} queries ...')
with torch.no_grad():
    q_sparse = model.encode_query(queries)

# Convert to scipy CSR
q_sparse = q_sparse.coalesce().cpu()
idx = q_sparse.indices().numpy()
vals = q_sparse.values().numpy()
q_csr = sp.csr_matrix(
    (vals, (idx[0], idx[1])),
    shape=tuple(q_sparse.shape),
    dtype=np.float32,
)
print('Queries encoded')

In [ ]:
# ── Sparse retrieval + adaptive cutoff ────────────────────────────────────
print('Scoring ...')
scores_matrix = (q_csr @ index.T).toarray().astype(np.float32)

top_k = min(RETRIEVAL_TOP_K, scores_matrix.shape[1])
top_idx = np.argpartition(scores_matrix, -top_k, axis=1)[:, -top_k:]


def adaptive_cutoff(cits, scores, gap_frac):
    if len(cits) <= 1:
        return cits
    threshold = gap_frac * float(scores[0])
    best_cut, best_gap = len(cits), 0.0
    for i in range(len(cits) - 1):
        gap = float(scores[i]) - float(scores[i + 1])
        if gap > threshold and gap > best_gap:
            best_gap, best_cut = gap, i + 1
    return cits[:best_cut]


rows = []
for i, (qid, idx_row) in enumerate(zip(test_df['query_id'], top_idx)):
    order = np.argsort(scores_matrix[i, idx_row])[::-1]
    sorted_idx = idx_row[order]
    cits = [citations[j] for j in sorted_idx]
    scrs = scores_matrix[i, sorted_idx]
    predicted = adaptive_cutoff(cits, scrs, ADAPTIVE_GAP_FRAC)
    rows.append({'query_id': qid, 'predicted_citations': ';'.join(predicted)})

submission = pd.DataFrame(rows)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('submission.csv saved!')
submission.head()